# 📅 2026-09-04(밤) ~ 09-05 개발 노트 : 점수 메커니즘 전면 재검토 — 검토 체인 넷이 놓친 결함, 절제 실측, 실패한 예측, 생애주기 분리

## 🎯 이틀의 목표 — "백필은 돌았다. 이제 그 데이터 위에서 점수가 맞는지를 증명한다"

- [x] 비용 오보고 정정 ($0.85 → $3.6 → **$2.45/회차** 대시보드 실측, 배치+캐시 할인 중복 확인)
- [x] 홀드아웃 150쌍 결과 → 49지표 교사급(MAE 0.77) / **gem_potential 사용 불가**(r=−0.022) → 리뷰 증거 지수로 교체 설계
- [x] 리뷰 8,653건 전량 수집 → 게이트 적용 → 활성 3,181건(37%). 신작 63%가 리뷰 9개 이하
- [x] 외부 검토 3건(TypingMind Fable 5 / GPT-5.6 Sol) + 자체 문서 → **넷이 전부 놓친 D-26** 을 소스에서 발견
- [x] 스냅샷 캐시 오염 차단(측정 신뢰성) → Day 1 폴백 수정 → score_v7(플래그) → 절제 도구
- [x] 절제 실측 2회: X-Factor 게이트 확정, **학생 스케일 보정 예측 실패 → 철회**
- [x] 신작·정착·유명작 생애주기 분리 + 랭킹 3(4)분할 + 신작 리그 (백엔드+프런트)
- [x] 2차 외부 검토 21건 코드 대조 → 전부 사실 → 전부 수정
- [x] 지표 63개 감사 → 합병 제안 → 개발자 반론으로 철회, 앵커 교정·신규 축으로 전환
- [ ] `ablation --pool default` 재실행 → v7 기본 전환 판단 (다음)
- [ ] gem 증거 지수 전환(R-3) → 노출 임계값 R-4 → 백필 재개 $27 → **push** (하루 넘게 미푸시, 약 40커밋)

> 이틀 배운 한 줄: **같은 문서를 본 검토자는 맹점도 같다. 근거는 코드와 실측뿐이고, 예측을 먼저 적어두지 않은 실험은 실험이 아니다.**


## ⭐ 1. 비용을 두 번 틀리게 보고했다 — 추정기가 아니라 대시보드

**문제:** 회차 비용을 $0.85 로 보고 → 사용자 결제 알림은 $28.88. 다시 $3.6 으로 수정하며 "배치 50% 와 프롬프트 캐시 90% 할인은 중복되지 않는다"고 단정.

**원인 두 겹:**
1. 코드 추정기(`usage_report`)가 미등록 모델을 gpt-4o-mini 단가로 계산 + 토큰을 문자/4 로 근사.
2. 할인 중복 여부를 **근거 없이** 단정. 대시보드 Cost 탭에 `batch api | cached input` 이 **별도 항목**으로 존재 — 중복 적용된다.

**실측 확정:** gpt-5.4-mini + 12-shot 배치 = **$0.0049/요청**, 500건 회차 $2.45, 8,653건 총 $43, 남은 5,500건 $27.
비용 구성 출력 71% / 캐시 입력 19% / 미적중 입력 11% → few-shot 축소는 절감 효과 미미(사용자도 품질 이유로 거부).

**교훈:** 사용자가 잘못된 전제로 돈을 썼다. 비용은 **1차 출처(대시보드)** 만 보고한다. 추정기는 등록된 단가만 쓰고 모르는 모델은 실패해야 한다 (`5da0104`, `6b1bd2b`).


## ⭐ 2. 홀드아웃 결과 — 49지표는 살고 gem 은 죽었다 → 추정을 관측으로

**측정:** 교사 게임 150건(gem 5분위 층화)을 학생 모델로 블라인드 재분석 → 같은 게임의 값 직접 비교.

| | 결과 | 판정 |
|---|---|---|
| 49 수치 지표 | 평균 MAE **0.77**, 46/49 가 1.5 이하, r>0.9 다수 | 거리(취향 매칭) 용도로 교사급 |
| 신뢰 낮은 3개 | modding_support r .43 / community_dependency .61 / monetization_fairness .36 | 임계값 용도 부적합 |
| gem_potential | MAE **19.5**, ρ 0.49, 편향 −18.4 | **사용 불가** |

**gem 을 보정으로 살릴 수 없었다:** 선형 보정(`calibrate_student`)을 맞추면 기울기 0.33 — 교사 홀드아웃 gem 이 40~95 에 몰려(인기작만 채점) 무명작 구간에 기준이 없는 **범위 제한**. 이 보정은 모든 신작을 교사 평균 근처로 밀어 넣는 퇴화 매핑이라 도구가 거부하게 만들었다(기울기 <0.7, 잔차 ≥ 전 MAE, 상수 예측 대비 개선 <15% → 거부).

**해결:** LLM 이 설명문에서 "숨은 명작일까"를 **추정**하는 대신 Steam 리뷰 **실측**으로: `gem_evidence = 100 × Wilson하한(긍정률, n, z=1.96) × 무명도(log 스케일, cap 20000, exp 0.5, floor 50)`.
- floor 50 이 없으면 리뷰 2~3개 100% 가 상위 독식(실측: 리뷰 2개 → 51.8). 하한을 두면 그 구간에서 Wilson 만 남아 "적을수록 유리" 역인센티브가 사라진다.
- LLM gem 과 증거 지수의 상관 **r = −0.022** — 둘은 다른 것을 재고 있었다.
- `--apply` 는 교사 코호트에 리뷰 근거가 100건 이상 없으면 거부(한 컬럼에 세 스케일 혼입 방지). 취소 시 `return 0` 이던 버그로 "n" 을 눌러도 0 이 써지던 사고 → `return 1`.

**교훈:** 검증셋의 **범위**를 먼저 봐야 한다. 교사가 본 적 없는 구간을 보정으로 만들어내면 그건 개선이 아니라 값 조작이다.


## ⭐ 3. 검토 체인 넷이 전부 놓친 결함 — D-26, 그리고 왜 놓쳤나

점수 메커니즘 as-is 문서(D-1~D-25, mermaid 3경로)를 써서 외부 모델 둘(Fable 5, GPT-5.6 Sol)에 검토를 받고, 정정 커밋을 다시 검토받았다. 세 검토는 유능했다 — 캐시 오염, `query_hint` 스키마 부재, hint_score 분모, σ 해석 오류를 잡았다.

그런데 모델을 바꿔(Fable 5.1) **문서가 아니라 `score_v6.py` 를 직접 읽자** 어디에도 없는 결함이 나왔다.

```python
for f in all_core_fields:                          # 장르 핵심 2~5개 + 의도 지표(v>=7)
    target_vec.append(target_metrics.get(f, 5.0))  # ← 장르 핵심은 preferences 에 없다 → 항상 5.0
```
호출부는 `target_metrics=preferences`(사용자가 입력한 3~4개 키만). 그러니 Core 75점의 65% 는 "사용자가 원한 것에 가깝나"가 아니라 **"이 게임이 자기 장르 핵심에서 얼마나 평범한가"** 를 재고 있었다.

공식 그대로 계산(힐링 cozy 9): 장르핵심 전부 5 → Core **66.1** / 전부 9 → **50.0** / cozy 6 + 전부 5 → **56.3**.
뛰어난 게임이 평범한 게임보다 16점 낮고, cozy 6 짜리 평범한 게임이 cozy 9 짜리 뛰어난 게임을 이긴다. 인디 장르 핵심이 `art_style_uniqueness` 라 **독창적 아트가 페널티** — 히든 인디 젬 사이트에서. 덤으로 `v >= 7` 임계는 0 만 버리는 게 아니라 7 미만 전부를 버려 힐링 프리셋 4개 중 3개(`time_pressure 1` 포함)가 점수에 안 들어갔다.

**왜 넷이 놓쳤나:** as-is 문서 §3 다이어그램이 `diff = (target − game) × weight` 라고만 그리고 **target 의 출처를 안 그렸다.** 같은 문서를 본 검토자는 맹점도 같다. 두 모델의 일치를 "교차검증"으로 기록한 것도 틀렸다 — 같은 입력의 일치는 근거가 아니다.

**함수 기본값이 채점 규칙이 된 사례.** `score_v6` 는 target 이 49차원 꽉 찬 dict(기준 게임)라는 전제로 짜였고, by-preference 는 sparse dict 를 넣었다. 계약 불일치를 `.get(f, 5.0)` 이 조용히 메웠다.

**조치:** `score_v7` — 사용자가 말한 지표만(임계 없음, 0 도 9 도 목표), `w = 1+|pref−5|/5`, 가중 RMSE `sqrt(Σw·diff²/Σw)`, `exp(−(d/3.5)²)` 로 거리 0 → 정확히 1.0, Core 93 + gem 6. 장르 핵심·X-Factor 제거. `SCORE_VERSION=v7` 플래그 뒤, 기본 v6. 로컬 검증: 위 사례 v7 은 93.0/93.0 (동일), time_pressure 1 vs 9 는 93.0/6.8 (v6 은 66.1/66.1 로 구분 못 함).

**교훈:** 외부 검토엔 요약이 아니라 **소스**를 넘긴다. 그리고 `.get(k, default)` 의 default 는 호출자가 그 키를 실제로 채우는지 확인해야 한다 — "안전한 중립"이 호출 맥락에선 채점 규칙이 된다 (`system_invariants.md` C-10).


## ⭐ 4. 측정 자체를 무효화하던 것 — 스냅샷 캐시 오염

점수 변경 전후를 비교하는 `rec_snapshot` 을 만들어 s0 을 찍었다. 검토가 지적: **캐시 키에 점수 로직 버전이 없다** (`semantic:{hash}:{limit}`, `rec:pref:{hash}:{count}`). s0 을 찍고 코드를 고쳐 s1 을 찍으면 s1 이 s0 의 캐시를 그대로 돌려받는다 — "변화 없음"이 안전의 증거가 아니라 **캐시가 살아있다는 증거**가 된다.

게다가 `invalidate_all()` 은 '키 0개'와 'Redis 예외'를 모두 0 으로 반환해 반환값으로 성공을 판정할 수 없었다.

**조치:** `--save` 마다 `POST /ops/cache/invalidate` → `GET /ops/cache` 로 `total_keys == 0` 확인(get_stats 는 실패 시 `{"error"}` 를 주므로 구분 가능) → 실패 시 스냅샷을 쓰지 않고 exit 2. 무효화 기록을 스냅샷 meta 에 박아 `--diff`/`--variance` 가 경고. 같은 라벨 덮어쓰기 거부, 16/16 성공 강제.
서빙 쪽엔 `CACHE_VERSION` prefix + 빠져 있던 조건들(`exclude_same_developer`, `required_tags`, `excluded_tags`, `min_gem_potential`, 나중엔 `score_version`, `include_new`) 을 키에 넣었다.

**부수 발견:** 상위 10 만 담는 스냅샷은 순위 영향력을 잴 수 없다. σ_final ≈ σ_core 이면 구성요소 간 음의 공분산이 크다는 뜻이고(ΣCov ≈ −0.55), 그건 게임의 성질이 아니라 **합계 상위 10 을 뽑았다는 선택이 만든 인공물**(collider). 내가 "σ 비중 = 랭킹 지배력" 이라 쓴 건 틀렸다 → 전체 풀 절제 도구가 필요했다.


## ⭐ 5. 절제 실측 — X-Factor 는 게이트이자 유명작 통로, 그리고 내 편향 표본 오류

`scripts/ablation.py`: 활성 풀 7,322건 전체를 채점하고 항을 하나씩 끄며 spearman / kendall / RBO@20 / 상위 20 유지 / 교사 비율 / 리뷰 중앙값.

| 발견 | 수치 |
|---|---|
| X-Factor 는 상수가 아니다 | 전체 풀에서 **X-F < 15.6 이 88.7%**, 18점 포화 7.7%. 상위 10 전원 15.6~18 은 전체의 11% 안에서만 뽑혔다는 뜻 — **게이트** |
| X-Factor 는 유명작을 올린다 | X-Factor 만 정렬: 상위 20 **교사 비율 0.90, 리뷰 중앙 19,700** (5 프리셋 동일 — 질의와 무관하니 당연). Core 만: 0.15~0.60 / 84~625 |
| 절단 확정 | Core σ 상위 10 = 1.14 → 전체 풀 11.6~16.0 |
| gem 은 지금 무력 | v6−Gem spearman 0.995+ |
| v7 방향 유지 + 상위권 재구성 | spearman 0.80~0.92, 상위 20 유지 0~4. 상위 5 취향 일치가 눈으로 v6 보다 낫다 (힐링: Spilled!/Chicory/Tiny Train — v6 는 Celebrity Memory 12in12/Minimalist Circuit) |

**내가 틀렸던 것:** 전날 홀드아웃 150건에서 X-Factor 100% 포화를 보고 "인기작 풀에서는 상수, 게이트 가설 강등"이라고 썼다. 홀드아웃은 **교사 인기작만** 담은 편향 표본 — 전체의 7.7% 만 보고 전체를 말했다. 검토자의 "강한 가설"이 맞았다. → 실수 분류 **G: 표본의 출처를 안 물음** 추가.

**교훈:** 검증에 쓴 표본이 모집단을 대표하는지를 **먼저** 묻는다. 그리고 상위 N 표본으로는 순위 영향력을 말할 수 없다 — 전체 풀을 돌려야 한다.


## ⭐ 6. 예측을 걸고 실험했고, 틀렸다 — 학생 스케일 보정 철회

v7 상위 20 의 교사 비율이 높았다(힐링 0.70, 액션 0.95, 기준선 0.57). 원인을 둘로 나눴다: (1) 같은 게임을 학생이 교사보다 낮게 주는 **스케일 편향**(홀드아웃 cozy +0.71, narrative +0.93 등 9개), (2) 인기 액션 게임이 실제로 reflex 9 인 **모집단 차이**(액션 프리셋 지표는 편향 ~0).

**예측을 먼저 적었다:** "보정 후 힐링·서사 교사비율 하락, 액션(편향 없음)은 0.9 유지. 액션까지 내려가면 분석이 틀린 것."

**결과:** 힐링 0.70→0.60(맞음, 단 Core 만은 0.60→0.85 상승) / 서사 0.50→0.50(틀림) / 액션 0.95(맞음) / **공포 0.85→1.00 악화**(예측 안 함).

**원인:** 절편 있는 보정 `cozy = +0.77 + 0.97x` 가 학생의 **0 을 0.77 로** 밀었다. 공포 프리셋은 `cozy 0` 이 목표 — 교사도 학생도 0 을 줬는데 보정 후 학생만 멀어져 상위 20 에서 전부 밀려났다. 평균 편향 +0.71 은 중간 구간에서 나온 것인데 절편은 0 에도 같은 이동을 강제한다. 원점 고정(y=bx)으로 바꿔도 cozy/humor/puzzle 의 전후 MAE 가 같다(0.83→0.83) — 비례 스케일도 아니다. 실체는 **바닥 효과**(교사 1~2 → 학생 0, cozy 0 비율 20.6% vs 35.7%). 선형 어떤 형태로도 못 고친다.

**조치:** `--revert` 로 9개 지표 × 8,653건 원본 복원. 도구엔 원점 고정 옵션만 남김(기준선용). 교사 비율은 Core 가 아니라 **발굴 층(gem 증거 + 히든젬 필터)** 이 처리한다 — PRD 의 역할 분담.

**교훈 (2차 검토가 정확히 짚음):** **교사 비율은 품질 목표가 아니다.** 두 코호트는 무작위 분할이 아니라 인기작/신작 모집단이라 비율 차이가 곧 편향이 아니다. 그걸 낮추는 최적화는 취향 적합도를 해친다 — 이 실패가 그 사례. 지표 보정 판단은 paired holdout 의 **구간별(0 / 1~3 / 4~6 / 7~10) 조건부 오차**로.


## 7. 신작·정착·유명작을 한 척도에 놓지 않는다 — 생애주기 분리와 신작 리그

**개발자 취지 원문:** "신작으로 신생 게임을 보호해서 그들만의 리그를 만들고 보여주자."

**설계:** 취향 일치(Core)는 모든 게임에 같은 척도. 발굴·랭킹은 생애주기별로 다른 질문.
- `new`(출시 ≤180일, **리뷰 수 무관** — 나이 우선) / `established` / `famous`(180일 초과 + 리뷰 ≥2만) / `upcoming`(미출시).
- 나이 우선인 이유 — 개발자 카나리아: "메챠 카멜레온(MECCHA CHAMELEON, 리뷰 8.7만)이 신작 랭킹 1위가 아니면 말이 안 된다." 리뷰 수로 먼저 나누면 '유명'으로 빠져 사라진다. 두 축이 필요해 `is_famous` 를 별도 필드로 — "신작 · 빠르게 검증됨".
- 메인 추천 기본 제외는 "신작"이 아니라 **근거 얇은 신작**(new & 리뷰 <100). 리뷰 수천 개는 데이터 부족이 아니다.
- **gem 은 established 만**(`gem_factor`). 신작은 시간이 없어서 무명이고, 유명작은 이미 발견됐다 — 발굴 질문 자체를 안 한다.
- 랭킹은 처음 만드는 것이었다 — 기존 `/ranking` 은 장르 프리셋 by-preference 결과였다. `GET /games/ranking?type=`: steady(Wilson×무명도) / rising(30일 상대 증가율 — `review_history` append-only 이력 필요) / new(리뷰/일) / new_quiet(리뷰 <100 중 평가 순 — 아직 조용한 신생 게임의 무대).
- 신작 리그(`new_only`): 같은 취향으로 **신작끼리만** 매칭한 섹션. 문구는 보호 취지로 — "판단 보류 중" 대신 "신작 리그 · 첫 리뷰 18건 · 첫 리뷰를 남겨보세요". "마음에 드는 게임에 첫 리뷰를 남기는 사람이 다음 히든젬을 만듭니다."
- 시점 사실: 수집이 3/16 이후라 **지금은 학생 코호트 거의 전부가 신작**. 9월 중순부터 매주 정착 구간으로 넘어간다.
- R-8: 슬라이더를 하나도 안 움직이면 프런트가 49개 전부 5.0 을 보내 "모든 축에서 평범한 게임"이 만점 — D-26 을 요청 쪽에서 재현하던 것. 중립값 제거 후 비면 400, 프런트는 랭킹으로 안내.


## ⭐ 8. 2차 외부 검토 21건 — 이번엔 소스를 읽었고 전부 맞았다

소스 원문·결정 기록·절제 실측을 폴더로 넘겼다(요약 문서는 의도적으로 제외). 21건을 코드로 하나씩 대조 — **사실 관계 오류 0.** 전부 수정.

| 급 | 지적 | 확인 |
|---|---|---|
| 치명 | 신작 리그에 기존 LLM gem 이 그대로 들어감 — 화면은 "발굴 점수 안 매김" | `new_only` 는 후보만 걸렀고 gem 은 A/B/C 전부 그대로 → `gem_factor` |
| 높음 | 0.1 반올림한 표시 점수로 정렬 + gem 을 타이브레이커로 재사용 → gem 이중 개입 | `raw_final_score` 분리, 정렬 `(raw, wilson, −reviews)` |
| 높음 | 절제 후보 풀(활성 전체)이 실제 기본 서빙 풀(리뷰<100 신작 제외)과 다름 → 30/31 회차는 "전체 풀 실험" | 입장 규칙 `lifecycle.admit()` 을 서빙 3경로와 절제 도구가 **공유**, `--pool default|include-new|new-only|hidden-gem|all` |
| 높음 | rising 에 famous 가 들어감 / `rank:*` 가 캐시 무효화에 없음 | 수정 |
| 중간 | 미래 출시일 → age 음수 → new / 지표명 검증 전에 중립 제거 → 잘못된 이름 조용히 통과 / SCORE_VERSION 만 바꾸면 옛 캐시 / identity 에 신뢰 불가 지표 / rising 이 모든 SQL 오류를 "수집 중"으로 위장 | 전부 수정 |
| 테스트 | positive 0 보존 테스트가 0.5 폴백 회귀를 못 잡음 / 단일 지표는 가중치가 상쇄돼 선형성 검증 불가 | `positive_used` 노출 / 두 지표 사례(≈3.221, 제곱이면 3.47) |

**내 프레임이 틀렸던 것 하나:** τ 는 순위를 **바꾸지 않는다** — match 가 단조 함수다. τ 와 gem 예산이 함께 정하는 것은 "gem 이 뒤집을 수 있는 취향 오차 폭"(τ 3.5·gem 6 → RMSE ≈ 0.9). gem 12 전환은 "오차 1.35 까지 발굴 근거로 상쇄한다"는 **정책 결정**이다. "상위권 폭으로 τ 조정"은 잘못된 프레임이었다.

**교훈:** 좋은 검토를 받는 조건 — 소스를 준다, 전제를 못 박는다, "동의는 한 줄·반대에 분량", 구체적 라인·공식 요구, 이미 아는 것 반복 금지. 그리고 받은 답은 **코드로 확인한 뒤에만** 기록한다.


## 9. 지표 63개 감사 — 데이터가 말한 것과 개발자가 바로잡은 것

교사 4,190 + 학생 8,760 원본으로 계산: PCA 누적 설명분산 50% = 4성분, 80% = 13, 90% = 21. 강한 중복 쌍 — reflex_demand~action_pacing **r .94**, lore~narrative .89, grind~growth .86(**정의 붕괴**: 반복 노동이 성장 보상과 같이 움직임), soundtrack~audio .85, learning_curve~difficulty_accessibility −.76. 설명문으로 잴 수 없는 7개(tutorial_quality 교사 σ **0.53**, save_flexibility .73, ui_ux_polish .78 …)는 정보가 아니라 기본값.

내 제안은 "6쌍 합병". **개발자 반론:** 느낌이 비슷해도 합치면 디테일이 떨어진다. 미래 게임에서 지표의 부재는 판단 공백 — A 축으로 매력적인 게임이 A 가 없어 저평가되거나 A+B+C 시너지에서 B 가 없어 반대로 고평가된다.

**받아들였다.** r=0.94 는 지금 모집단의 사실이지 의미 동일의 증명이 아니다 — 리듬 게임(reflex 9 / pacing 5), 느린 호러(time_pressure 8 / pacing 2)처럼 갈리는 게임이 "세분화된 지표 사이트"가 잡아야 하는 게임이다. 그리고 **v7 마스크에선 중복이 점수를 해치지 않는다** — 사용자가 고른 축만 들어간다. 중복이 문제였던 건 49개를 전부 넣던 v6 시절.

**바뀐 조치:** 합병 대신 **앵커를 갈라놓는다**(프롬프트 v2 에 두 축이 다르게 나오는 예시: "Elden Ring = lore 10 / narrative 4", "Hades = growth 9 / grind 3"). 실행 품질 7개는 계속 수집하되 슬라이더에 "추정치" 표시. 신규 축 추가 — `romance_focus`, `mystery_factor`, `reading_load`, 배경 설정 4종, `has_deckbuilding`, `has_life_sim`. 그리고 LLM 이 아니라 크롤러로 채울 실측 — **한국어 지원**(가장 큰 공백), 코옵 방식 3종, 컨트롤러, 성인 콘텐츠, `playtime_forever` 중앙값(appreviews 응답에 이미 있다). 구성: 취향 축 52 + 속성 17 + 실측 9.

**교훈:** 상관 분석은 "무엇을 합칠까"가 아니라 "LLM 이 무엇을 구분하지 못하고 있나"를 알려주는 도구다. 답은 축을 없애는 게 아니라 **구분을 가르치는 것**이었다.


## 📋 다음 할 일

**측정·전환 (순서 고정):**
- ⬜ `docker compose restart fastapi` → `pytest tests/test_score_invariants.py tests/test_lifecycle.py -v`
- ⬜ `python -m scripts.ablation --pool default` (실제 기본 서빙 풀) → v7 기본 전환 판단. `--pool new-only` 로 신작 리그 확인
- ⬜ `refresh_reviews --recheck` 한 번 → `review_history` 생성 (요즘 뜨는 = 주간 갱신 4~5회 뒤)
- ⬜ `rec_snapshot --save s2_day1` → `--diff s1 s2`
- ⬜ gem 증거 지수 전환(R-3): 컬럼 분리 + too_new/famous 상태 + 폴백·review_bonus·confidence 제거 + gem 예산 12(= 취향 오차 1.35 상쇄 정책) → 절제 재실행
- ⬜ `SCORE_VERSION=v7` → `--save s3` → 상위 5 눈검사 + 장르 방향 유지 확인
- ⬜ 노출 게이트 R-4(리뷰 ≥3 & Wilson ≥.35) 적용 — 메인은 근거 얇은 신작을 기본 제외하므로 품질 영향 없음

**데이터:**
- ⬜ 크롤러 확장: supported_languages(한국어) / categories(코옵·컨트롤러) / content_descriptors / playtime 중앙값
- ⬜ 프롬프트 v2: 앵커 분리(reflex vs pacing, lore vs narrative, grind vs growth, learning vs accessibility) + 신규 축 3+6 → 전 게임 학생 모델로 통일 부여(≈$25~30), 백필 재개($27)와 같은 창
- ⬜ Vibe secondary 목표값 72개 검증(대표 20 + 반례 20) 후 `VIBE_SECONDARY_ENABLED`
- ⬜ '메챠 카멜레온'(4864560, 리뷰 209) vs 'MECCHA CHAMELEON'(4704690, 8.7만) 중복/데모 확인

**운영:**
- ⬜ **`git push`** — 하루 넘게 미푸시(프로젝트 ~40커밋 + div-log). Vercel/Railway 자동 배포 → 프런트 랭킹·신작 리그·토글, 백엔드 Day 1 폴백 수정·랭킹 API 가 나간다. v7 은 플래그 꺼짐이라 점수 로직은 v6 그대로
- ⬜ 포트폴리오 ⑫⑬⑭ 반영 완료 → 수치 모음 갱신
